# EEG motor imagery: left fist vs right fist

A deliberately simple baseline: can we tell from EEG alone whether a person imagines moving their **left** or **right** fist?

- **Data:** PhysioNet EEG Motor Movement/Imagery dataset, subjects 1 to 10, runs 4, 8 and 12 (about 45 left/right trials per subject).
- **How to run:** `Run All`. The first run downloads the recordings, which takes a few minutes.
- **Note:** subject 1 was used to choose the method, so the fairest average excludes it.

In [202]:
import numpy as np
import mne
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


In [203]:
runs = [4, 8, 12]          # runs with imagined left/right fist movement
channels = ["C3..", "Cz..", "C4.."]    # sensors over the motor areas(names end with ".." in this dataset)
subjects = range(1,11)

## Pipeline for one subject

1. Load the 3 recordings and join them, then band-pass filter 1-40 Hz to remove drift and noise
2. Turn the annotations (T0 = rest, T1 = left fist, T2 = right fist) into events and cut 4-second epochs
3. Keep the 8-30 Hz band (mu and beta rhythms, linked to movement)
4. Summarize each epoch as the log-variance of 3 channels (C3, Cz, C4)
5. Keep left vs right only (drop the rest epochs)
6. Standardize the features, classify with logistic regression, and score with 5-fold cross-validation repeated 10 times

In [204]:
def evaluer(sujet):
    # Load and join the recordings, then filter 1-40 Hz
    fichiers = mne.datasets.eegbci.load_data(subjects=sujet, runs=[4, 8, 12])
    raws = [mne.io.read_raw_edf(f, preload=True) for f in fichiers]
    raw = mne.concatenate_raws(raws)
    raw.filter(l_freq=1, h_freq=40)

    # Events from annotations, then 4-second epochs
    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id, tmin=0, tmax=4, baseline=None, preload=True)

    # Keep 8-30 Hz, then log-variance per channel
    epochs_mb = epochs.copy().filter(l_freq=8, h_freq=30)
    y = epochs.events[:, 2] # 1 = rest, 2 = left fist, 3 = right fist
    features = np.log(epochs_mb.get_data().var(axis=2))

    colonnes = [epochs.ch_names.index(nom) for nom in ["C3..", "Cz..", "C4.."]]
    features = features[:, colonnes]

    #  Left vs right only
    masque = y != 1
    features, y = features[masque], y[masque]
    # Standardize, classify, score with repeated cross-validation
    chaine = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)
    return cross_val_score(chaine, features, y, cv=cv).mean()

In [205]:
mne.set_log_level('WARNING')
resultat ={}
for s in range(1,11):
    resultat[s]=evaluer(s)
moyen = sum(resultat.values())/len(resultat)
print(resultat)
print(moyen)

{1: np.float64(0.6288888888888889), 2: np.float64(0.6088888888888888), 3: np.float64(0.37999999999999995), 4: np.float64(0.7844444444444444), 5: np.float64(0.4599999999999999), 6: np.float64(0.4355555555555555), 7: np.float64(0.7644444444444443), 8: np.float64(0.5533333333333332), 9: np.float64(0.6244444444444445), 10: np.float64(0.6688888888888888)}
0.5908888888888887
